# GNNs for Systemic Financial Risk — Colab runner

Runtime → *Change runtime type* → **T4 GPU**. Datasets are generated on the fly (seconds) and cached on the local Colab disk; results and checkpoints go to Google Drive so they survive session resets.

Run the cells top to bottom. Each experiment cell can be re-run on its own after a disconnect (re-run the setup cells first).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [3]:
# Get the code: EITHER clone from GitHub (set REPO_URL) OR copy the project folder you uploaded to Drive.
REPO_URL = ''   # e.g. 'https://github.com/<you>/<repo>.git'  (private repos need a token)
DRIVE_COPY = '/content/drive/MyDrive/sysrisk'   # used when REPO_URL is empty

import os, shutil, subprocess
if os.path.exists('/content/sysrisk'):
    shutil.rmtree('/content/sysrisk')
if REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/sysrisk'], check=True)
else:
    shutil.copytree(DRIVE_COPY, '/content/sysrisk', ignore=shutil.ignore_patterns('data', 'results', '*.pdf', '.git'))
%cd /content/sysrisk

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/content/drive/MyDrive/sysrisk'

In [4]:
!pip install -q torch_geometric
!pip install -q -e . --no-deps
import torch, torch_geometric
print('torch', torch.__version__, '| pyg', torch_geometric.__version__, '| cuda', torch.cuda.is_available())


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: file:///C:/Users/Sangameshwar/OneDrive/Desktop/Ai%20in%20finance/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


ModuleNotFoundError: No module named 'torch'

In [ ]:
import os
os.environ['SYSRISK_DATA_DIR'] = '/content/data'                 # fast local disk
RESULTS = '/content/drive/MyDrive/sysrisk_results'               # persistent
os.makedirs(RESULTS, exist_ok=True)
%env RESULTS={RESULTS}

## 1. Sanity checks
Unit tests (solver identities, paper-2 stylised networks, equivariance) and the paper-2 generator check: mean S should be ≈ 262 (ER) and ≈ 238 (CP). CPf uses a different fixed matrix than the paper, so its mean differs.

In [ ]:
!python -m pytest -q -p no:cacheprovider -W ignore
!python -W ignore scripts/generate_data.py --n-graphs 2500 | head -60

## 2. E1 — in-distribution benchmark (N = 100)
Each config trains all baselines + G1 (paper GNN) + G2 (EN-aligned GNN) with 5 seeds. Add `--seeds 0` for a quicker first pass.

In [ ]:
!python scripts/run_experiment.py --config configs/e1_er.yaml  --out $RESULTS/runs
!python scripts/run_experiment.py --config configs/e1_cp.yaml  --out $RESULTS/runs
!python scripts/run_experiment.py --config configs/e1_cpf.yaml --out $RESULTS/runs

## 3. E2 — size generalisation
Train at N = 100, test at N ∈ {50, 200, 500, 1000} (`fd_*` = fixed expected degree, `fp_*` = fixed link probability as in the mid-report).

In [ ]:
!python scripts/run_experiment.py --config configs/e2_size_er.yaml       --out $RESULTS/runs
!python scripts/run_experiment.py --config configs/e2_size_cp.yaml       --out $RESULTS/runs
!python scripts/run_experiment.py --config configs/e2_size_er_mixed.yaml --out $RESULTS/runs

## 4. E5 — runtime and tables

In [ ]:
!python scripts/benchmark_runtime.py --out $RESULTS
!python scripts/make_tables.py --runs $RESULTS/runs --out $RESULTS/tables